In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, log_loss, classification_report
import pickle
import warnings
warnings.filterwarnings('ignore')

def train_model(fifa_ranking_path, results_path, model_output_path='best_model.pkl', preprocessor_output_path='preprocessor.pkl'):

    # 1. Load Data
    fifa_ranking_df = pd.read_csv(fifa_ranking_path)
    results_df = pd.read_csv(results_path)

    # 2. Match Results Data - Nettoyage & Calculs de base
    results_df.dropna(subset=['home_score', 'away_score'], inplace=True)
    results_df['date'] = pd.to_datetime(results_df['date'])
    results_df = results_df.sort_values(by=['date']).reset_index(drop=True)

    results_df['goal_difference'] = results_df['home_score'] - results_df['away_score']
    results_df['match_outcome'] = results_df['goal_difference'].apply(lambda x: 1 if x > 0 else (-1 if x < 0 else 0))

    # --- CALCUL PROPRE DE LA FORME GLOBALE ---
    home_side = results_df[['date', 'home_team', 'home_score', 'away_score', 'match_outcome']].rename(
        columns={'home_team': 'team', 'home_score': 'goals_for', 'away_score': 'goals_against', 'match_outcome': 'outcome'}
    )
    away_side = results_df[['date', 'away_team', 'away_score', 'home_score', 'match_outcome']].rename(
        columns={'away_team': 'team', 'away_score': 'goals_for', 'home_score': 'goals_against'}
    )
    away_side['outcome'] = -results_df['match_outcome'] # Inverser l'issue pour l'extérieur

    all_games = pd.concat([home_side, away_side]).sort_values(['team', 'date']).reset_index(drop=True)

    # Calcul des rolling features globales (Shift 1 pour éviter la fuite de données)
    window = 10
    all_games['avg_goals_scored'] = all_games.groupby('team')['goals_for'].transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    all_games['avg_goals_conceded'] = all_games.groupby('team')['goals_against'].transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    all_games['avg_outcome'] = all_games.groupby('team')['outcome'].transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    all_games['avg_goal_diff'] = all_games.groupby('team').apply(lambda x: (x['goals_for'] - x['goals_against']).shift(1).rolling(window, min_periods=1).mean()).reset_index(level=0, drop=True)

    # On renomme "team" en "home_team" pour la fusion domicile
    team_stats_home = all_games[['date', 'team', 'avg_goals_scored', 'avg_goals_conceded', 'avg_outcome', 'avg_goal_diff']].rename(
        columns={'team': 'home_team', 'avg_goals_scored': 'home_avg_scored', 'avg_goals_conceded': 'home_avg_conceded', 'avg_outcome': 'home_avg_outcome', 'avg_goal_diff': 'home_avg_gdiff'}
    )
    # On renomme "team" en "away_team" pour la fusion extérieur
    team_stats_away = all_games[['date', 'team', 'avg_goals_scored', 'avg_goals_conceded', 'avg_outcome', 'avg_goal_diff']].rename(
        columns={'team': 'away_team', 'avg_goals_scored': 'away_avg_scored', 'avg_goals_conceded': 'away_avg_conceded', 'avg_outcome': 'away_avg_outcome', 'avg_goal_diff': 'away_avg_gdiff'}
    )

    # 3. FIFA Ranking Preprocessing
    fifa_ranking_df.dropna(subset=['rank'], inplace=True)
    fifa_ranking_df['rank_date'] = pd.to_datetime(fifa_ranking_df['rank_date'])
    fifa_ranking_df = fifa_ranking_df.sort_values(by='rank_date')

    # 4. Fusions Temporelles Strictes (direction='backward')
    # A] Association de la Forme Récente
    results_df = pd.merge_asof(results_df.sort_values('date'), team_stats_home.sort_values('date'), on='date', by='home_team', direction='backward')
    results_df = pd.merge_asof(results_df.sort_values('date'), team_stats_away.sort_values('date'), on='date', by='away_team', direction='backward')

    # B] Association des Rangs FIFA
    fifa_home = fifa_ranking_df[['rank_date', 'country_full', 'rank', 'total_points']].rename(columns={'country_full': 'home_team', 'rank': 'home_rank', 'total_points': 'home_points'})
    fifa_away = fifa_ranking_df[['rank_date', 'country_full', 'rank', 'total_points']].rename(columns={'country_full': 'away_team', 'rank': 'away_rank', 'total_points': 'away_points'})

    merged_df = pd.merge_asof(results_df.sort_values('date'), fifa_home.sort_values('rank_date'), left_on='date', right_on='rank_date', by='home_team', direction='backward').drop(columns=['rank_date'])
    merged_df = pd.merge_asof(merged_df.sort_values('date'), fifa_away.sort_values('rank_date'), left_on='date', right_on='rank_date', by='away_team', direction='backward').drop(columns=['rank_date'])

    # Target
    merged_df['match_outcome'] = merged_df['match_outcome'].replace({1: 0, 0: 1, -1: 2}) # 0: Domicile, 1: Nul, 2: Extérieur
    merged_df.dropna(subset=['home_rank', 'away_rank'], inplace=True)
    merged_df.fillna(0, inplace=True)

    # 5. Features selection
    features_num = [
        'home_rank', 'away_rank', 'home_points', 'away_points',
        'home_avg_scored', 'home_avg_conceded', 'home_avg_outcome', 'home_avg_gdiff',
        'away_avg_scored', 'away_avg_conceded', 'away_avg_outcome', 'away_avg_gdiff'
    ]
    features_cat = ['neutral']
    
    X = merged_df[features_num + features_cat]
    y = merged_df['match_outcome'].astype(int)

    # 6. SPLIT CHRONOLOGIQUE (Sécurisé)
    cutoff = int(len(X) * 0.8)
    X_train, X_test = X.iloc[:cutoff], X.iloc[cutoff:]
    y_train, y_test = y.iloc[:cutoff], y.iloc[cutoff:]

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), features_num),
            ('cat', OneHotEncoder(handle_unknown='ignore'), features_cat)
        ]
    )

    X_train_processed = preprocessor.fit_transform(X_train)
    X_test_processed = preprocessor.transform(X_test)

    # 7. Training
    gbc_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42)
    gbc_model.fit(X_train_processed, y_train)

    # Evaluation réelle (Crash Test)
    preds = gbc_model.predict(X_test_processed)
    probas = gbc_model.predict_proba(X_test_processed)
    
    print(f"--- RÉSULTATS DU SCRIPT CORRIGÉ ---")
    print(f"Accuracy réelle en Test : {accuracy_score(y_test, preds):.4f}")
    print(f"Log-loss réel en Test   : {log_loss(y_test, probas):.4f}\n")
    print(classification_report(y_test, preds, target_names=['Domicile', 'Nul', 'Extérieur']))

    # Sauvegarde
    with open(model_output_path, 'wb') as f: pickle.dump(gbc_model, f)
    with open(preprocessor_output_path, 'wb') as f: pickle.dump(preprocessor, f)

if __name__ == '__main__':
    train_model('fifa_ranking-2024-06-20.csv', 'results.csv')

--- RÉSULTATS DU SCRIPT CORRIGÉ ---
Accuracy réelle en Test : 0.5957
Log-loss réel en Test   : 0.8940

              precision    recall  f1-score   support

    Domicile       0.62      0.86      0.72      2280
         Nul       0.37      0.05      0.09      1140
   Extérieur       0.57      0.61      0.59      1410

    accuracy                           0.60      4830
   macro avg       0.52      0.51      0.47      4830
weighted avg       0.55      0.60      0.53      4830

